In [ ]:
# enable cuda devices
import os
import warnings
import math

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

os.chdir('/app')
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import time
import logging
import numpy as np
import torch
import json
import torch.nn as nn
import torchvision.transforms as transforms
from utils.common_utils import process_config
from datetime import datetime
from utils.strip_utils import compute_strip_importances, model_strip_group
from models import dataset
from utils.bit_config import bit_config_dict
from utils import misc

In [ ]:

def train(train_loader, model, criterion, optimizer, epoch, configs):
    model.eval()
    for i, (images, target) in enumerate(train_loader):

        images = images.cuda(0, non_blocking=True)
        target = target.cuda(0, non_blocking=True)

        # compute output
        output = model(images)
        loss = criterion(output, target)
        
        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def validate(val_loader, model, criterion, configs):
    losses = AverageMeter('Loss', ':.4e')
    top1 = AverageMeter('Acc@1', ':6.2f')
    top5 = AverageMeter('Acc@5', ':6.2f')

    model.eval()

    with torch.no_grad():
        for i, (images, target) in enumerate(val_loader):
            images = images.cuda(0, non_blocking=True)
            target = target.cuda(0, non_blocking=True)

            # compute output
            output = model(images)
            loss = criterion(output, target)

            # measure accuracy and record loss
            acc1, acc5 = accuracy(output, target, topk=(1, 5))
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0], images.size(0))
            top5.update(acc5[0], images.size(0))
            
    return top1.avg

def accuracy(output, target, topk=(1,)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self, name, fmt=':f'):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)

In [ ]:
def hessian_trace(model, val_loader, criterion, configs):
    importances, strip_importances_per_layer = compute_strip_importances(model, val_loader, criterion)
    strip_group = model_strip_group(model, importances, strip_importances_per_layer, configs.bits, configs.ratio, log=False)
    return strip_group

def strip_wise_hessian(configs):
    inference_log_dir = os.path.join(configs.logdir, configs.net, configs.dataset, 'best.pth')
    net = configs.net

    #--------------------------------------------------------------------------------------------------
    if configs.dataset == 'cifar10':
        _, val_loader = dataset.get_cifar10(batch_size=configs.batch_size)
        num_classes=10
    elif configs.dataset == 'cifar100':
        _, val_loader = dataset.get_cifar100(batch_size=configs.batch_size)
        num_classes=100
    else:
        raise ValueError("Unknown dataset type")

    #--------------------------------------------------------------------------------------------------
    if net == 'resnet18':
        from models.ResNet import resnet18
        model = resnet18(num_classes)
        model.load_state_dict(torch.load(inference_log_dir))    
    elif net == 'resnet50':
        from models.ResNet import resnet50
        model = resnet50(num_classes)
        model.load_state_dict(torch.load(inference_log_dir))
    elif net == 'vgg11':
        from models.VGG import vgg11
        model = vgg11(num_classes)
        model.load_state_dict(torch.load(inference_log_dir))
    elif net == 'vgg19':
        from models.VGG import vgg19
        model = vgg19(num_classes)
        model.load_state_dict(torch.load(inference_log_dir))
    else:
        raise ValueError("Unknown model type")

    t_begin = time.time()

    torch.cuda.set_device(0)
    model = model.cuda(0)
    criterion = nn.CrossEntropyLoss().cuda(0)

    strip_group = hessian_trace(model, val_loader, criterion, configs)

    return strip_group

In [ ]:
def fine_tuning(configs, strip_group):
    net = configs.net
    inference_log_dir = os.path.join(configs.logdir, configs.net, configs.dataset, 'best.pth')

    #--------------------------------------------------------------------------------------------------
    if configs.dataset == 'cifar10':
        train_loader, val_loader = dataset.get_cifar10(batch_size=configs.batch_size)
        num_classes=10
    elif configs.dataset == 'cifar100':
        train_loader, val_loader = dataset.get_cifar100(batch_size=configs.batch_size)
        num_classes=100
    else:
        raise ValueError("Unknown dataset type")

    #--------------------------------------------------------------------------------------------------
    if net == 'vgg11':
        from models.VGG import vgg11
        pre_trained_model = vgg11()
        pre_trained_model.load_state_dict(torch.load(inference_log_dir))
        from models.Q_VGG import q_vgg11
        model = q_vgg11(pre_trained_model)

    elif net == 'resnet18':
        from models.ResNet import resnet18
        pre_trained_model = resnet18()
        pre_trained_model.load_state_dict(torch.load(inference_log_dir))
        from models.Q_ResNet import q_resnet18
        model = q_resnet18(pre_trained_model)

    elif net == 'resnet50':
        from models.ResNet import resnet50
        pre_trained_model = resnet50()
        pre_trained_model.load_state_dict(torch.load(inference_log_dir))
        from models.Q_ResNet import q_resnet50
        model = q_resnet50(pre_trained_model)

    else:
        raise ValueError("Unknown model type")
    #--------------------------------------------------------------------------------------------------

    bit_config = strip_group

    #--------------------------------------------------------------------------------------------------
    name_counter = 0
    for name, m in model.named_modules():
        if name in bit_config.keys():
            name_counter += 1
            setattr(m, 'quant_mode', 'symmetric')
            setattr(m, 'bias_bit', configs.bias_bit)
            setattr(m, 'quantize_bias', (configs.bias_bit != 0))
            setattr(m, 'per_strip', True)
            setattr(m, 'name', f"Conv_{name_counter}")
            bitwidth = bit_config[name]
            setattr(m, 'weight_bit', bitwidth)

    #--------------------------------------------------------------------------------------------------

    torch.cuda.set_device(0)
    model = model.cuda(0)

    criterion = nn.CrossEntropyLoss().cuda(0)

    optimizer = torch.optim.SGD(model.parameters(), configs.lr,
                                momentum=configs.momentum,
                                weight_decay=configs.weight_decay)


    best_acc1 = 0
    for epoch in range(configs.epochs):
        # adjust_learning_rate(optimizer, epoch, configs)

        train(train_loader, model, criterion, optimizer, epoch, configs)

        acc1 = validate(val_loader, model, criterion, configs)

        # remember best acc@1 and save checkpoint
        is_best = acc1 > best_acc1
        best_acc1 = max(acc1, best_acc1)

    return best_acc1


In [ ]:
def accuracy_objective_function(configs, ratio):
    configs.ratio = ratio
    strip_group = strip_wise_hessian(configs)
    accuracy = fine_tuning(configs, strip_group)

    return accuracy

def approximate_slope(configs, ratio, epsilon=0.025):
    alpha = accuracy_objective_function(configs, ratio)
    beta = accuracy_objective_function(configs, ratio + epsilon)
    print(f'Compression:{math.floor(ratio * 1000) / 1000.0 } \
                Best acc: {math.floor(alpha * 100) / 100.0}')
    print(f'Compression:{math.floor((ratio + epsilon) * 1000) / 1000.0 } \
                Best acc: {math.floor(beta * 100) / 100.0}')

    grad = (alpha - beta)
    slope = grad / epsilon
    return slope

In [ ]:
import math
import argparse
import numpy as np
import matplotlib.pyplot as plt

configs = process_config('/app/configs/exp_for_cifar/iteration.json')

compression_ratio = 0.1
lr = 0.05
iterations = 18

ratio = compression_ratio
history = []

current_slope = approximate_slope(configs, ratio)

for _ in range(1, iterations):
    next_slope = approximate_slope(configs, ratio + lr)

    with open('./acc_record.txt', 'a') as file:
        file.write(f'{math.floor(ratio * 100) / 100.0} => {current_slope} \n')

    ratio = ratio + lr
    current_slope = next_slope